In [1]:
import pandas as pd
import sqlite3
df = pd.read_csv("cleaned_global_ecommerce_sales.csv")
conn = sqlite3.connect("ecommerce.db")
df.to_sql("sales", conn, if_exists="replace", index=False)

1810

Monthly Sales Trend

In [3]:
query = """
SELECT strftime('%Y-%m', order_date) AS month,
SUM(total_sales) AS total_sales
FROM sales
GROUP BY month
ORDER BY month;
"""

pd.read_sql(query, conn)

,month,total_sales
0,2023-01,4686.68
1,2023-02,6841.23
2,2023-03,6628.42
3,2023-04,6381.33
4,2023-05,5873.13
5,2023-06,9068.11
6,2023-07,5806.54
7,2023-08,9546.67
8,2023-09,9076.38
9,2023-10,8080.46


Top 10 Customers

In [4]:
query = """
SELECT customer_name,
SUM(total_sales) AS total_sales
FROM sales
GROUP BY customer_name
ORDER BY total_sales DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,customer_name,total_sales
0,Sakura Thomas,1163.41
1,Karen Suzuki,972.28
2,Oscar Martinez,967.27
3,Wei Harris,960.81
4,Olga Nilsson,850.96
5,Ingrid Zhang,801.09
6,Diego Kim,790.03
7,Elizabeth Johnson,783.03
8,Jessica Müller,781.96
9,Thomas Chen,743.00


Top 10 Products

In [5]:
query = """
SELECT product_name,
SUM(total_sales) AS total_sales
FROM sales
GROUP BY product_name
ORDER BY total_sales DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,product_name,total_sales
0,Mesh Back Task Chair,14794.78
1,Portable External SSD 1TB,13159.11
2,Bookshelf 5-Tier,12115.49
3,Floating Wall Shelves Set,11767.30
4,Folding Table Portable,11449.28
5,Webcam HD 1080p,10927.15
6,Mechanical Gaming Keyboard,10308.45
7,Wireless Ergonomic Mouse,9995.39
8,Wireless Bluetooth Headphones,9849.31
9,Travel Backpack 30L,9699.77


Category-wise Sales

In [8]:
query = """
SELECT product_category,
SUM(total_sales) AS total_sales
FROM sales
GROUP BY product_category;
"""

pd.read_sql(query, conn)

,product_category,total_sales
0,Clothing & Accessories,57730.36
1,Furniture,90424.83
2,Office Supplies,19390.81
3,Technology,93738.78


Average Sales by Category

In [10]:
query = """
SELECT product_category,
AVG(total_sales) AS average_sales
FROM sales
GROUP BY product_category;
"""

pd.read_sql(query, conn)

,product_category,average_sales
0,Clothing & Accessories,144.687619
1,Furniture,239.853660
2,Office Supplies,37.798850
3,Technology,179.920883


Highest Sale

In [11]:
query = """
SELECT *
FROM sales
ORDER BY total_sales DESC
LIMIT 1;
"""

pd.read_sql(query, conn)

,order_id,order_date,customer_name,customer_segment,country,region,product_category,product_name,quantity,unit_price,discount_percent,total_sales,shipping_cost,profit,payment_method
0,ORD-11422,2023-09-03,David Clark,Consumer,Germany,Europe,Furniture,Folding Table Portable,10,67.1,10,603.9,14.47,186.83,Cash on Delivery


Lowest Sale

In [12]:
query = """
SELECT *
FROM sales
ORDER BY total_sales ASC
LIMIT 1;
"""

pd.read_sql(query, conn)

,order_id,order_date,customer_name,customer_segment,country,region,product_category,product_name,quantity,unit_price,discount_percent,total_sales,shipping_cost,profit,payment_method
0,ORD-11001,2025-02-20,Isabella Jackson,Corporate,Japan,Asia Pacific,Office Supplies,Paper Clips Box 500pc,1,3.03,20,2.42,11.16,-10.1,Cash on Delivery


Create a View

In [17]:
conn.execute("DROP VIEW IF EXISTS top_sales;")
conn.execute("""
CREATE VIEW top_sales AS
SELECT *
FROM sales
WHERE total_sales > 500;
""")

In [18]:
pd.read_sql("SELECT order_id, order_date, customer_name, customer_segment, country, region, product_category, product_name, quantity, unit_price, discount_percent, total_sales, shipping_cost, profit, payment_method FROM top_sales;", conn)

,order_id,order_date,customer_name,customer_segment,country,region,product_category,product_name,quantity,unit_price,discount_percent,total_sales,shipping_cost,profit,payment_method
0,ORD-10402,2023-01-14,Jamal Schmidt,Consumer,Mexico,North America,Furniture,Folding Table Portable,6,96.58,0,579.48,10.22,221.57,Credit Card
1,ORD-10594,2023-02-21,Ayumi Khan,Consumer,United States,North America,Technology,Smart LED Desk Lamp,8,72.89,0,583.12,14.27,248.13,Credit Card
2,ORD-10915,2023-03-08,Ahmed Harris,Consumer,Brazil,South America,Furniture,Floating Wall Shelves Set,6,99.69,0,598.14,18.01,221.25,Credit Card
3,ORD-10676,2023-03-17,Wei Chen,Consumer,Saudi Arabia,Middle East & Africa,Technology,Mechanical Gaming Keyboard,3,174.46,0,523.38,14.65,220.87,Credit Card
4,ORD-10071,2023-04-05,Richard Müller,Consumer,United States,North America,Furniture,Ergonomic Office Chair,2,252.68,0,505.36,8.12,194.02,PayPal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,ORD-11608,2025-11-01,Isabella Sato,Consumer,Argentina,South America,Clothing & Accessories,Travel Backpack 30L,6,92.98,0,557.88,18.67,260.27,PayPal
63,ORD-11256,2025-11-03,Ivan Sato,Consumer,Spain,Europe,Furniture,Corner L-Shaped Desk,2,297.84,15,506.33,8.55,140.37,Bank Transfer
64,ORD-11940,2025-11-03,Hanna Hassan,Home Office,France,Europe,Technology,Portable External SSD 1TB,6,110.54,10,596.92,21.12,211.02,PayPal
65,ORD-11693,2025-12-10,Carlos Rodriguez,Corporate,Brazil,South America,Furniture,Bookshelf 5-Tier,3,194.57,0,583.71,13.80,219.68,Credit Card


Query Execution Plan

In [20]:
pd.read_sql("""
EXPLAIN QUERY PLAN
SELECT *
FROM sales
WHERE total_sales > 500;
""", conn)

,id,parent,notused,detail
0,2,0,0,SCAN sales


Close the Connection

In [21]:
conn.close()